In [1]:
import pandas as pd
import numpy as np
import shap
import joblib

# Load original clean dataset
df = pd.read_csv("5-star edible sushi.csv")
df["report_month"] = pd.to_datetime(df["report_month"])

# Load Phase 4 checkpoint
checkpoint = joblib.load("phase4_checkpoint.joblib")

# Restore objects
cost_final_model = checkpoint["cost_final_model"]
schedule_final_model = checkpoint["schedule_final_model"]

cost_shap_features = checkpoint["cost_shap_features"]
schedule_shap_features = checkpoint["schedule_shap_features"]

cost_shap_values = checkpoint["cost_shap_values"]
schedule_shap_values = checkpoint["schedule_shap_values"]

cost_feature_names = checkpoint["cost_feature_names"]
schedule_feature_names = checkpoint["schedule_feature_names"]

example_project = checkpoint["example_project"]
example_date = checkpoint["example_date"]

print("✅ Phase 4 objects loaded.")
print("Cost model:", type(cost_final_model))
print("Schedule model:", type(schedule_final_model))

C:\Users\vatsh\OneDrive\Desktop\ml\meowwww\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Phase 4 objects loaded.
Cost model: <class 'sklearn.pipeline.Pipeline'>
Schedule model: <class 'sklearn.pipeline.Pipeline'>


In [2]:
# ==========================================
# RECREATE FINAL MODEL DATA OBJECTS
# ==========================================

cost_preprocessor = cost_final_model.named_steps["preprocessor"]
cost_xgb = cost_final_model.named_steps["model"]

schedule_preprocessor = schedule_final_model.named_steps["preprocessor"]
schedule_xgb = schedule_final_model.named_steps["model"]

# Same cost test split used previously
cost_mask = df["target_cost_overrun_12m"].notna()

cost_test_mask = (
    cost_mask &
    (df["report_month"] >= pd.Timestamp("2021-01-01"))
)

# Same schedule test split
schedule_mask = df["target_schedule_risk_12m"].notna()

schedule_test_mask = (
    schedule_mask &
    (df["report_month"] >= pd.Timestamp("2021-01-01"))
)

# Columns excluded from ML
drop_cols = [
    "project_id",
    "project_name",
    "report_month",
    "target_cost_overrun_12m",
    "target_schedule_risk_12m",
    "cost_target_valid",
    "serial_no",
    "source_file",
    "source_page",
    "implementing_agency"
]

drop_cols = [
    c for c in drop_cols
    if c in df.columns
]

X = df.drop(columns=drop_cols)

X_test = X.loc[cost_test_mask].copy()
X_schedule_test = X.loc[schedule_test_mask].copy()

print("Cost test:", X_test.shape)
print("Schedule test:", X_schedule_test.shape)

Cost test: (7393, 47)
Schedule test: (4825, 47)


In [3]:
# ==========================================
# SHAP HELPERS
# ==========================================

def clean_shap_feature_name(name):

    if name.startswith("num__"):
        return name.replace("num__", "", 1)

    if name.startswith("cat__"):
        remainder = name.replace("cat__", "", 1)
        return remainder.split("_", 1)[0]

    return name


FEATURE_LABELS = {
    "project_age": "project age",
    "anticipated_cost_percentage": "anticipated cost percentage",
    "exp_vs_anti_prct": "expenditure vs anticipated cost",
    "exp_vs_org_prct": "expenditure vs original cost",
    "sector_overrun_rate": "historical sector overrun rate",
    "agency_overrun_rate": "historical agency overrun rate",
    "anticipated_cost_crore": "anticipated project cost",
    "original_cost_crore": "original project cost",
    "current_cost": "current project cost",
    "cumulative_expenditure_crore": "cumulative expenditure",
    "original_duration_months": "original project duration",
    "duration_overrun_months": "duration overrun",
    "project_duration_elapsed_percentage": "project duration elapsed",
    "anticipated_delay_from_original": "anticipated delay from original schedule",
    "remaining_org_months": "remaining original schedule",
    "schedule_change_12m": "12-month schedule change",
    "expenditure_growth_12m": "12-month expenditure growth",
    "milestone_completion_percentage": "milestone completion percentage",
    "milestones_total": "total milestones",
    "approval_year": "approval year",
    "year": "report year",
    "month": "report month"
}

print("✅ SHAP helpers ready.")

✅ SHAP helpers ready.


In [4]:
# ==========================================
# PHASE 4.3 — SHAP VS SELECTED FEATURES
# ==========================================

selected_features = set([
    "project_age",
    "original_duration_months",
    "duration_overrun_months",
    "anticipated_cost_percentage",
    "exp_vs_anti_prct",
    "exp_vs_org_prct",
    "current_cost",
    "cumulative_expenditure_crore",
    "project_duration_elapsed_percentage",
    "anticipated_delay_from_original",
    "remaining_org_months",
    "delay_revised_months",
    "milestone_completion_percentage",
    "cost_revision_percentage",
    "expenditure_growth_3m",
    "expenditure_growth_6m",
    "expenditure_growth_12m",
    "cost_growth_3m",
    "cost_growth_6m",
    "cost_growth_12m",
    "schedule_change_3m",
    "schedule_change_6m",
    "schedule_change_12m",
    "milestone_progress_change_3m",
    "milestone_progress_change_6m",
    "milestone_progress_change_12m",
    "expenditure_progress_gap_org",
    "expenditure_progress_gap_anti",
    "cost_rebaseline_signal",
    "schedule_progress_mismatch",
    "agency_overrun_rate",
    "sector_overrun_rate"
])

cost_top = set(
    cost_shap_features
    .sort_values("mean_abs_shap", ascending=False)
    .head(20)["feature"]
)

schedule_top = set(
    schedule_shap_features
    .sort_values("mean_abs_shap", ascending=False)
    .head(20)["feature"]
)

print("COST — TOP SHAP FEATURES IN SELECTED SET:")
print(sorted(cost_top & selected_features))

print("\nSCHEDULE — TOP SHAP FEATURES IN SELECTED SET:")
print(sorted(schedule_top & selected_features))

print("\nCOST — TOP SHAP FEATURES OUTSIDE SELECTED SET:")
print(sorted(cost_top - selected_features))

print("\nSCHEDULE — TOP SHAP FEATURES OUTSIDE SELECTED SET:")
print(sorted(schedule_top - selected_features))

COST — TOP SHAP FEATURES IN SELECTED SET:
['anticipated_cost_percentage', 'anticipated_delay_from_original', 'cumulative_expenditure_crore', 'current_cost', 'exp_vs_anti_prct', 'exp_vs_org_prct', 'expenditure_growth_12m', 'expenditure_growth_3m', 'expenditure_growth_6m', 'original_duration_months', 'project_age', 'project_duration_elapsed_percentage', 'sector_overrun_rate']

SCHEDULE — TOP SHAP FEATURES IN SELECTED SET:
['agency_overrun_rate', 'anticipated_delay_from_original', 'cumulative_expenditure_crore', 'duration_overrun_months', 'exp_vs_anti_prct', 'exp_vs_org_prct', 'expenditure_growth_12m', 'milestone_completion_percentage', 'original_duration_months', 'project_age', 'project_duration_elapsed_percentage', 'remaining_org_months', 'schedule_change_12m', 'sector_overrun_rate']

COST — TOP SHAP FEATURES OUTSIDE SELECTED SET:
['anticipated_cost_crore', 'approval_year', 'month', 'original_cost_crore', 'sector', 'year']

SCHEDULE — TOP SHAP FEATURES OUTSIDE SELECTED SET:
['anticipate

In [5]:
# ==========================================
# FINAL BACKEND-READY EXPLANATION
# ==========================================

def explain_project(project_id, report_month, top_n=5):

    report_month = pd.Timestamp(report_month)

    source_df = df[
        (df["project_id"] == project_id) &
        (df["report_month"] == report_month)
    ].copy()

    if len(source_df) == 0:
        return {"error": "Project/date combination not found."}

    drop_cols = [
        "project_id",
        "project_name",
        "report_month",
        "target_cost_overrun_12m",
        "target_schedule_risk_12m",
        "cost_target_valid",
        "serial_no",
        "source_file",
        "source_page",
        "implementing_agency"
    ]

    drop_cols = [
        c for c in drop_cols
        if c in source_df.columns
    ]

    X_local = source_df.drop(columns=drop_cols)

    def explain_model(model, preprocessor, xgb):

        X_transformed = preprocessor.transform(X_local)

        probability = model.predict_proba(X_local)[0, 1]

        explainer = shap.TreeExplainer(xgb)

        shap_values = explainer.shap_values(
            X_transformed
        )[0]

        feature_names = (
            preprocessor.get_feature_names_out()
        )

        drivers = pd.DataFrame({
            "transformed_feature": feature_names,
            "shap_value": shap_values,
            "abs_shap": np.abs(shap_values)
        })

        drivers["feature"] = [
            clean_shap_feature_name(x)
            for x in feature_names
        ]

        drivers = drivers.sort_values(
            "abs_shap",
            ascending=False
        )

        return probability, drivers

    cost_probability, cost_drivers = explain_model(
        cost_final_model,
        cost_preprocessor,
        cost_xgb
    )

    schedule_probability, schedule_drivers = explain_model(
        schedule_final_model,
        schedule_preprocessor,
        schedule_xgb
    )

    def format_drivers(drivers):

        result = []

        for _, row in drivers.head(top_n).iterrows():

            feature = row["feature"]

            label = FEATURE_LABELS.get(
                feature,
                feature.replace("_", " ")
            )

            result.append({
                "feature": label,
                "direction": (
                    "increases risk"
                    if row["shap_value"] > 0
                    else "reduces risk"
                ),
                "shap_value": round(
                    float(row["shap_value"]), 4
                )
            })

        return result

    return {
        "project_id": project_id,
        "report_month": str(report_month.date()),

        "cost_overrun": {
            "probability": round(
                float(cost_probability), 4
            ),
            "top_drivers": format_drivers(
                cost_drivers
            )
        },

        "schedule_risk": {
            "probability": round(
                float(schedule_probability), 4
            ),
            "top_drivers": format_drivers(
                schedule_drivers
            )
        }
    }

print("✅ explain_project() ready.")

✅ explain_project() ready.


In [6]:
# ==========================================
# TEST BACKEND-READY EXPLANATIONS
# ==========================================

test_examples = (
    df.loc[
        X_test.index,
        ["project_id", "report_month"]
    ]
    .drop_duplicates()
    .head(5)
)

for _, row in test_examples.iterrows():

    result = explain_project(
        row["project_id"],
        row["report_month"],
        top_n=5
    )

    print("\n" + "=" * 70)
    print("PROJECT:", result["project_id"])
    print("DATE:", result["report_month"])

    print(
        "COST RISK:",
        result["cost_overrun"]["probability"]
    )

    print(
        "SCHEDULE RISK:",
        result["schedule_risk"]["probability"]
    )

    print("\nCOST DRIVERS:")
    for d in result["cost_overrun"]["top_drivers"]:
        print(
            f" • {d['feature']} → {d['direction']}"
        )

    print("\nSCHEDULE DRIVERS:")
    for d in result["schedule_risk"]["top_drivers"]:
        print(
            f" • {d['feature']} → {d['direction']}"
        )


PROJECT: 180100078
DATE: 2021-01-01
COST RISK: 0.8508
SCHEDULE RISK: 0.0002

COST DRIVERS:
 • project age → increases risk
 • duration overrun → increases risk
 • approval year → increases risk
 • sector → increases risk
 • expenditure vs anticipated cost → reduces risk

SCHEDULE DRIVERS:
 • anticipated delay from original schedule → reduces risk
 • remaining original schedule → reduces risk
 • duration overrun → reduces risk
 • project age → reduces risk
 • cumulative expenditure → reduces risk

PROJECT: 180100078
DATE: 2021-02-01
COST RISK: 0.8278
SCHEDULE RISK: 0.0002

COST DRIVERS:
 • project age → increases risk
 • duration overrun → increases risk
 • approval year → increases risk
 • expenditure vs anticipated cost → reduces risk
 • sector → increases risk

SCHEDULE DRIVERS:
 • anticipated delay from original schedule → reduces risk
 • remaining original schedule → reduces risk
 • duration overrun → reduces risk
 • project age → reduces risk
 • cumulative expenditure → reduces r

In [7]:
# ==========================================
# PHASE 4 — FINAL EXIT CHECK
# ==========================================

phase4_status = {
    "Global SHAP — Cost Model": "COMPLETED",
    "Global SHAP — Schedule Model": "COMPLETED",
    "Local SHAP Explanations": "COMPLETED",
    "SHAP Feature Cross-Check": "COMPLETED",
    "Backend explain_project() Function": "COMPLETED",
    "Both Models Explained": "YES"
}

print("=" * 60)
print("PHASE 4 — EXPLAINABILITY")
print("=" * 60)

for item, status in phase4_status.items():
    print(f"{item:<45} {status}")

print("=" * 60)
print("PHASE 4 STATUS: COMPLETE ✅")
print("=" * 60)

PHASE 4 — EXPLAINABILITY
Global SHAP — Cost Model                      COMPLETED
Global SHAP — Schedule Model                  COMPLETED
Local SHAP Explanations                       COMPLETED
SHAP Feature Cross-Check                      COMPLETED
Backend explain_project() Function            COMPLETED
Both Models Explained                         YES
PHASE 4 STATUS: COMPLETE ✅


In [8]:
# ==========================================
# PHASE 4 — SUMMARY
# ==========================================

print("""
PHASE 4 — EXPLAINABILITY SUMMARY

1. Global SHAP importance was calculated for both
   Cost Overrun and Schedule Risk models.

2. Local SHAP explanations were implemented to identify
   the strongest drivers behind individual predictions.

3. SHAP importance was cross-checked against the
   previously selected feature set.

4. A backend-ready explain_project(project_id, report_month)
   function was implemented.

5. The explanation output contains:
   - Cost overrun probability
   - Schedule risk probability
   - Top risk drivers
   - Direction of each driver's effect

Both predictive models are now explainable at both
global and individual-project levels.

PHASE 4: COMPLETE ✅
""")


PHASE 4 — EXPLAINABILITY SUMMARY

1. Global SHAP importance was calculated for both
   Cost Overrun and Schedule Risk models.

2. Local SHAP explanations were implemented to identify
   the strongest drivers behind individual predictions.

3. SHAP importance was cross-checked against the
   previously selected feature set.

4. A backend-ready explain_project(project_id, report_month)
   function was implemented.

5. The explanation output contains:
   - Cost overrun probability
   - Schedule risk probability
   - Top risk drivers
   - Direction of each driver's effect

Both predictive models are now explainable at both
global and individual-project levels.

PHASE 4: COMPLETE ✅

